# PyTorch 网络层详解 - 综合应用

本 Notebook 是系列教程的第5部分，展示包含所有高级技巧的完整训练流程。

**包含内容**：
- 完整模型定义（带残差连接的 CNN）
- 混合精度训练 (AMP)
- 梯度累积
- 梯度裁剪
- 学习率调度器
- Early Stopping
- 模型保存与加载
- Hook 机制
- 训练过程可视化
- 模型导出 (TorchScript)

> [<a href="http://127.0.0.1:5500/html/04-9_pipeline_01_flow.html" target="_blank">👉 在浏览器打开 训练闭环与Hook图解</a>](http://127.0.0.1:5500/html/04-9_pipeline_01_flow.html)


In [ ]:
# ============================================================
# 1. 导入必要的库
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler
import torch.utils.checkpoint as checkpoint
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# 解决中文乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "WenQuanYi Micro Hei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

#### 1. 准备数据

In [ ]:
# ============================================================
# 准备模拟数据
# ============================================================

num_samples = 2000
num_classes = 10

# 生成模拟图像数据（CIFAR-10 风格）
X_train = torch.randn(num_samples, 3, 32, 32)
y_train = torch.randint(0, num_classes, (num_samples,))
X_val = torch.randn(200, 3, 32, 32)
y_val = torch.randint(0, num_classes, (200,))

# 创建 DataLoader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, 
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False,
                        num_workers=2, pin_memory=True)

print("数据准备完成")
print(f"  训练集: {len(train_dataset)} 样本")
print(f"  验证集: {len(val_dataset)} 样本")

#### 2. 模型定义（ResNet 风格）

In [ ]:
# ============================================================
# 定义 ResNet 风格的 CNN 模型
# ============================================================

class ResidualBlock(nn.Module):
    """残差块：Conv → BN → ReLU → Conv → BN → (+ 残差) → ReLU"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = self.relu(out)
        return out


class ResNetCIFAR(nn.Module):
    """用于 CIFAR-10 的 ResNet 风格网络"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)
        
        self.layer1 = ResidualBlock(32, 32)
        self.layer2 = ResidualBlock(32, 64, stride=2)
        self.layer3 = ResidualBlock(64, 128, stride=2)
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

model = ResNetCIFAR()
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

#### 3. 训练配置

In [ ]:
# ============================================================
# 训练配置
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 损失函数：带标签平滑
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 优化器：AdamW
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)

# 学习率调度：OneCycleLR
max_epochs = 10
total_steps = len(train_loader) * max_epochs
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=total_steps)

# 混合精度
scaler = GradScaler()

# Early Stopping
best_val_acc = 0.0
patience = 3
early_stop_counter = 0

train_losses, val_accs = [], []

print(f"设备: {device}")
print(f"训练步数: {total_steps}")

#### 4. Hook 机制（特征提取演示）

In [ ]:
# ============================================================
# 前向 Hook：提取中间层特征
# ============================================================

features = {}

def forward_hook(module, input, output):
    features['layer3_output'] = output.detach().cpu()

# 在 layer3 上注册 Hook
hook = model.layer3.register_forward_hook(forward_hook)

# 执行一次前向传播
sample_x = torch.randn(1, 3, 32, 32).to(device)
with torch.no_grad():
    _ = model(sample_x)

print(f"Hook 提取的特征: {features['layer3_output'].shape}")
print("  用途：中间层特征提取、可视化")

# 移除 Hook
hook.remove()
print("  Hook 已移除")

#### 5. 训练循环

In [ ]:
# ============================================================
# 训练循环（含 AMP、梯度裁剪）
# ============================================================

print("\n" + "=" * 70)
print("开始训练")
print("=" * 70)

for epoch in range(max_epochs):
    # ---------- 训练阶段 ----------
    model.train()
    epoch_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        
        # 混合精度前向传播
        with autocast():
            loss = criterion(model(batch_x), batch_y)
        
        # 反向传播
        scaler.scale(loss).backward()
        
        # 梯度裁剪
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 优化器更新
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss.item() * batch_x.size(0)
    
    avg_loss = epoch_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)
    
    # ---------- 验证阶段 ----------
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model(batch_x).argmax(1)
            correct += (pred == batch_y).sum().item()
    
    val_acc = correct / len(val_loader.dataset)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}/{max_epochs}, Loss: {avg_loss:.4f}, Acc: {val_acc:.4f}")
    
    # ---------- Early Stopping ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        early_stop_counter = 0
        print(f"  ✅ 保存最佳模型，acc: {best_val_acc:.4f}")
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print(f"  ⏹ Early stopping at epoch {epoch+1}")
            break

# 加载最佳模型
if os.path.exists('best_model.pth'):
    best_model = ResNetCIFAR().to(device)
    best_model.load_state_dict(torch.load('best_model.pth'))
    print(f"✅ 最佳验证准确率: {best_val_acc:.4f}")
    os.remove('best_model.pth')

#### 6. 训练过程可视化

In [ ]:
# ============================================================
# 可视化训练过程
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss')
ax1.set_title('训练损失曲线')
ax1.grid(True, alpha=0.3)

ax2.plot(val_accs, marker='o', color='orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('验证准确率曲线')
ax2.axhline(y=best_val_acc, color='red', linestyle='--', label=f'Best: {best_val_acc:.3f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f"训练完成！最佳验证准确率: {best_val_acc:.4f}")

#### 7. 模型导出（TorchScript）

In [ ]:
# ============================================================
# TorchScript 导出
# ============================================================

# 加载最佳模型
if 'best_model' in dir():
    # 导出为 TorchScript
    example_input = torch.randn(1, 3, 32, 32).to(device)
    best_model.eval()
    
    # 使用 trace 方式导出
    traced_model = torch.jit.trace(best_model, example_input)
    traced_model.save('model_traced.pt')
    print("✅ TorchScript 模型已保存: model_traced.pt")
    
    # 验证导出结果
    loaded_model = torch.jit.load('model_traced.pt')
    with torch.no_grad():
        output1 = best_model(example_input)
        output2 = loaded_model(example_input)
    
    diff = (output1 - output2).abs().max().item()
    print(f"  原始模型与导出模型输出差异: {diff:.6f}")
    print(f"  差异 { '< 1e-5' if diff < 1e-5 else '> 1e-5'}，导出成功 ✅")
    
    # 清理
    if os.path.exists('model_traced.pt'):
        os.remove('model_traced.pt')
        print("  已清理: model_traced.pt")

#### 8. 训练最佳实践检查清单

#### ✅ 数据加载
- [x] `pin_memory=True` 加速数据加载
- [x] `num_workers` 多进程加载

#### ✅ 模型定义
- [x] 合理的权重初始化（Kaiming）
- [x] 残差连接
- [x] BatchNorm 使用

#### ✅ 训练配置
- [x] 学习率调度器（OneCycleLR）
- [x] 梯度裁剪（防止梯度爆炸）
- [x] 混合精度训练（AMP）
- [x] 标签平滑（防止过拟合）

#### ✅ 模型管理
- [x] Early Stopping
- [x] 保存最佳模型
- [x] Hook 机制（特征提取）

#### ✅ 部署
- [x] TorchScript 导出

---

#### 9. 系列总结

**第1部分：总论** - 核心概念、有状态 vs 无状态、Parameter vs Buffer

**第2部分：有参数层** - 卷积、线性、归一化、嵌入、序列层、Transformer

**第3部分：无参数层** - 激活函数、池化、Dropout

**第4部分：容器** - Sequential、ModuleList、ModuleDict、ParameterList/ParameterDict

**第5部分：综合应用** - 完整训练流程 + 高级技巧

通过这5个 Notebook，你已掌握了 PyTorch 网络层的完整知识体系，从基础概念到工业级训练实践。